# 04 — Horizon analysis

Casts west-facing rays from each observer and measures when the eclipsed Sun clears the reconstructed skyline.


In [1]:
# Load the horizon-analysis modules, event window, rasters, and observer inputs.
import json
import re
import xml.etree.ElementTree as ET

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.features import geometry_mask
from rasterio.mask import mask
from rasterio.windows import from_bounds
from shapely.geometry import Point, Polygon

from eclipse_viewshed.project import repository_root
PROJECT_ROOT = repository_root()

from eclipse_viewshed import aoi, horizon, solar, surface

INTERIM = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
EXTERNAL = PROJECT_ROOT / "data" / "external"
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED.mkdir(parents=True, exist_ok=True)

DSM_VRT = INTERIM / "dsm_1m.vrt"
DTM_VRT = INTERIM / "dtm_1m.vrt"
CLASSES = INTERIM / "classes_1m.tif"
for path in (DSM_VRT, DTM_VRT, CLASSES):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} missing - run notebooks 02 and 03 first"
        )

# Load the event window produced by notebook 01 and sample the solar path.
RAY_M = aoi.DEFAULT_RAY_M
contact_data = json.loads(
    (PROCESSED / "01_eclipse_contacts.json").read_text()
)
C1_CEST = contact_data["first_contact_cest"]
MAX_CEST = contact_data["maximum_cest"]
C4_CEST = contact_data["last_contact_cest"]
ECLIPSE_WINDOW = (C1_CEST, C4_CEST)

track = solar.sun_track(step_seconds=5)
tmax = solar.at_time(track, MAX_CEST)


## Eye elevations


In [2]:
# Resolve each observer's eye elevation from its configured vertical reference.
observers = pd.read_csv(EXTERNAL / "observers.csv")
observers[["x_l72", "y_l72"]] = observers.apply(
    lambda row: pd.Series(aoi.to_lambert72(row["lat"], row["lon"])),
    axis=1,
)

eye_rows = []
for _, observer in observers.iterrows():
    z_eye, base = horizon.resolve_eye_z(
        observer["x_l72"],
        observer["y_l72"],
        DSM_VRT,
        DTM_VRT,
        z_mode=observer["z_mode"],
        eye_height_m=observer["eye_height_m"],
        abs_eye_z_taw=observer.get("abs_eye_z_taw"),
    )
    eye_rows.append({
        "name": observer["name"],
        "z_mode": observer["z_mode"],
        "base_taw": round(base, 2),
        "eye_height_m": observer["eye_height_m"],
        "z_eye_taw": round(z_eye, 2),
    })

eye = pd.DataFrame(eye_rows)
observers["z_eye_taw"] = eye["z_eye_taw"]
eye


,name,z_mode,base_taw,eye_height_m,z_eye_taw
0,Nieuw Zuid,ground,6.85,1.6,8.45
1,Scheldekaaien Zuid,ground,6.80,1.6,8.40
2,"Wandelterras, Het Steen",ground,6.99,6.6,13.59
3,Droogdokkenpark,ground,4.30,1.6,5.90
4,MAS panoramic platform,surface,68.68,1.6,70.28


## Placement check


In [3]:
# Measure the local surroundings and flag observer coordinates that are poorly placed.
rows = []
for _, o in observers.iterrows():
    s = horizon.describe_surroundings(
        o["x_l72"], o["y_l72"], DSM_VRT, DTM_VRT, CLASSES, radius_m=300)
    rows.append({
        "observer": o["name"],
        "ground_taw": s["ground_taw"],
        "nearest water (m)": s.get("nearest_water_m"),
        "water within 100 m": f"{s.get('water_within_100m_pct', 0):.0f}%",
        "nearest obstruction (m)": s.get("nearest_obstruction_m"),
        "tallest within 50 m": s["tallest_within_50m"],
        "tallest within 200 m": s["tallest_within_200m"],
        "at point": surface.CLASS_LABELS.get(s.get("class_at_point"), "?"),
    })

placement = pd.DataFrame(rows)

# Flag coordinates that are not close to the river.
ON_BANK_WATER_M = 60

placement["near the river"] = [
    (w is not None and w <= ON_BANK_WATER_M)
    for w in placement["nearest water (m)"]
]
placement


,observer,ground_taw,nearest water (m),water within 100 m,nearest obstruction (m),tallest within 50 m,tallest within 200 m,at point,near the river
0,Nieuw Zuid,6.88,0,52%,40,8.1,20.3,water,True
1,Scheldekaaien Zuid,6.87,13,43%,16,7.8,29.2,ground,True
2,"Wandelterras, Het Steen",6.98,6,46%,0,41.2,41.2,vegetation,True
3,Droogdokkenpark,4.29,0,39%,16,7.9,24.8,water,True
4,MAS panoramic platform,7.03,22,32%,0,64.2,64.2,building,True


## New development on Linkeroever

Checks how new development on Linkeroever since the DSM and DTM were produced affects eclipse viewing.


In [4]:
# Download the current OpenStreetMap buildings and read their recorded heights and floor counts.
LINKEROEVER_CHANGED_WGS84 = [
    (4.348138581, 51.214769755),
    (4.3732440574, 51.214769755),
    (4.3732440574, 51.2205042788),
    (4.348138581, 51.2205042788),
]
CONSERVATIVE_BUILDING_HEIGHT_M = 16.0
changed_area = Polygon([
    aoi.to_lambert72(lat, lon)
    for lon, lat in LINKEROEVER_CHANGED_WGS84
])

osm_dir = RAW / "osm"
osm_dir.mkdir(parents=True, exist_ok=True)
osm_path = osm_dir / "linkeroever_changed_area.osm"
if not osm_path.exists():
    west, south = LINKEROEVER_CHANGED_WGS84[0]
    east, north = LINKEROEVER_CHANGED_WGS84[2]
    response = requests.get(
        "https://api.openstreetmap.org/api/0.6/map",
        params={"bbox": f"{west},{south},{east},{north}"},
        headers={"User-Agent": "eclipse-viewshed-2026/1.0"},
        timeout=120,
    )
    response.raise_for_status()
    osm_path.write_bytes(response.content)

osm_root = ET.fromstring(osm_path.read_bytes())
osm_nodes = {
    node.attrib["id"]: (float(node.attrib["lon"]), float(node.attrib["lat"]))
    for node in osm_root.findall("node")
}

def first_number(value):
    match = re.search(r"[-+]?\d+(?:[.,]\d+)?", value or "")
    return float(match.group().replace(",", ".")) if match else np.nan

osm_rows = []
for way in osm_root.findall("way"):
    tags = {tag.attrib["k"]: tag.attrib["v"] for tag in way.findall("tag")}
    if "building" not in tags:
        continue
    refs = [node.attrib["ref"] for node in way.findall("nd")]
    coords = [osm_nodes[ref] for ref in refs if ref in osm_nodes]
    if len(coords) < 4 or coords[0] != coords[-1]:
        continue
    levels = first_number(tags.get("building:levels"))
    tagged_height = first_number(tags.get("height"))
    osm_rows.append({
        "osm_id": int(way.attrib["id"]),
        "name": tags.get("name", ""),
        "floors": levels,
        "recorded height (m)": tagged_height,
        "geometry": Polygon(coords),
    })

osm_buildings = gpd.GeoDataFrame(osm_rows, geometry="geometry", crs=4326).to_crs(31370)
osm_buildings = osm_buildings[osm_buildings.geometry.intersects(changed_area)].copy()
osm_buildings["conservative height (m)"] = pd.concat([
    pd.Series(CONSERVATIVE_BUILDING_HEIGHT_M, index=osm_buildings.index),
    osm_buildings["floors"] * 4,
    osm_buildings["recorded height (m)"],
], axis=1).max(axis=1)
recorded_floor_counts = osm_buildings["floors"].dropna()
MAX_BUILDING_HEIGHT_M = float(osm_buildings["conservative height (m)"].max())
pd.DataFrame([{
    "OpenStreetMap building footprints": len(osm_buildings),
    "footprints with floor counts": int(osm_buildings["floors"].notna().sum()),
    "highest recorded floor count": (
        int(recorded_floor_counts.max()) if not recorded_floor_counts.empty else None
    ),
    "minimum height used for every footprint (m)": CONSERVATIVE_BUILDING_HEIGHT_M,
    "highest height used (m)": MAX_BUILDING_HEIGHT_M,
}])


,OpenStreetMap building footprints,footprints with floor counts,highest recorded floor count,minimum height used for every footprint (m),highest height used (m)
0,596,29,6,16.0,24.0


In [5]:
# Locate the newer buildings relative to Scheldekaaien Zuid and the Sun at maximum eclipse.
site = observers.loc[observers["name"] == "Scheldekaaien Zuid"].iloc[0]
corner_x, corner_y = np.array(changed_area.exterior.coords).T
bearings, _ = aoi.bearing_distance(
    site["x_l72"], site["y_l72"], corner_x, corner_y
)
nearest_range_m = Point(site["x_l72"], site["y_l72"]).distance(changed_area)
with rasterio.open(DTM_VRT) as src:
    changed_ground = mask(
        src, [changed_area.__geo_interface__], crop=True, filled=False
    )[0]
highest_ground_taw = float(changed_ground.max())
highest_roof_taw = highest_ground_taw + MAX_BUILDING_HEIGHT_M
upper_bound_angle = float(np.degrees(np.arctan2(
    highest_roof_taw - site["z_eye_taw"]
    - horizon.curvature_drop(nearest_range_m),
    nearest_range_m,
)))

# Add every current building footprint in the changed area to the Scheldekaaien Zuid skyline.
development_azimuths = np.arange(
    solar.WEDGE_AZ_MIN, solar.WEDGE_AZ_MAX + horizon.DEFAULT_AZ_STEP / 2,
    horizon.DEFAULT_AZ_STEP,
)
development_horizon = np.full(development_azimuths.shape, -90.0)
development_distance = np.full(development_azimuths.shape, np.nan)
development_height = np.full(development_azimuths.shape, np.nan)

frames = []
for path in (PROJECT_ROOT / "data" / "raw" / "grb").glob("*.geojson"):
    frame = gpd.read_file(path, bbox=changed_area.bounds, engine="pyogrio")
    if not frame.empty:
        frames.append(frame)
current_buildings = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True), geometry="geometry", crs=frames[0].crs
).drop_duplicates(subset=["geometry"])
current_buildings = current_buildings[
    current_buildings.geometry.intersects(changed_area)
].copy()
current_buildings.geometry = current_buildings.geometry.intersection(changed_area)

building_models = [
    (footprint, CONSERVATIVE_BUILDING_HEIGHT_M)
    for footprint in current_buildings.geometry
]
building_models.extend(
    (row.geometry, row["conservative height (m)"])
    for _, row in osm_buildings.iterrows()
    if row["conservative height (m)"] > CONSERVATIVE_BUILDING_HEIGHT_M
)

with rasterio.open(DTM_VRT) as src:
    for footprint, building_height_m in building_models:
            window = from_bounds(*footprint.bounds, src.transform)
            window = window.round_offsets().round_lengths()
            ground = src.read(1, window=window, masked=True)
            transform = src.window_transform(window)
            inside = geometry_mask(
                [footprint.__geo_interface__], ground.shape, transform, invert=True
            )
            valid = inside & ~np.ma.getmaskarray(ground)
            if not valid.any():
                continue
            rows, cols = np.where(valid)
            xs, ys = rasterio.transform.xy(transform, rows, cols, offset="center")
            azimuths, distances = aoi.bearing_distance(
                site["x_l72"], site["y_l72"], np.asarray(xs), np.asarray(ys)
            )
            roof_taw = (
                float(np.ma.median(ground[valid]))
                + building_height_m
            )
            angles = np.degrees(np.arctan2(
                roof_taw - site["z_eye_taw"] - horizon.curvature_drop(distances),
                distances,
            ))
            bins = np.rint(
                (azimuths - solar.WEDGE_AZ_MIN) / horizon.DEFAULT_AZ_STEP
            ).astype(int)
            keep = (bins >= 0) & (bins < development_azimuths.size)
            for bin_index in np.unique(bins[keep]):
                choices = np.flatnonzero(keep & (bins == bin_index))
                choice = choices[np.argmax(angles[choices])]
                if angles[choice] > development_horizon[bin_index]:
                    development_horizon[bin_index] = angles[choice]
                    development_distance[bin_index] = distances[choice]
                    development_height[bin_index] = roof_taw

assert upper_bound_angle < float(tmax["altitude_deg"])
pd.DataFrame([{
    "site tested": site["name"],
    "direction to newer buildings": f"{bearings.min():.1f}–{bearings.max():.1f}°",
    "building footprints tested": len(current_buildings),
    "minimum assumed building height (m)": CONSERVATIVE_BUILDING_HEIGHT_M,
    "highest assumed building height (m)": MAX_BUILDING_HEIGHT_M,
    "closest edge (m)": round(nearest_range_m),
    "highest possible skyline (degrees)": round(upper_bound_angle, 2),
    "Sun height at maximum (degrees)": round(float(tmax["altitude_deg"]), 2),
    "maximum eclipse visible": upper_bound_angle < float(tmax["altitude_deg"]),
}])


,site tested,direction to newer buildings,building footprints tested,minimum assumed building height (m),highest assumed building height (m),closest edge (m),highest possible skyline (degrees),Sun height at maximum (degrees),maximum eclipse visible
0,Scheldekaaien Zuid,276.4–311.7°,585,16.0,24.0,1116,1.71,7.7,True


In [6]:
# Download the current FPC footprints, assign roof heights, and test them against every skyline.
FPC_OSM_BBOX = (4.3605, 51.2062, 4.3675, 51.2102)
fpc_path = osm_dir / "antwerp_fpc.osm"
if not fpc_path.exists():
    response = requests.get(
        "https://api.openstreetmap.org/api/0.6/map",
        params={"bbox": ",".join(map(str, FPC_OSM_BBOX))},
        headers={"User-Agent": "eclipse-viewshed-2026/1.0"},
        timeout=120,
    )
    response.raise_for_status()
    fpc_path.write_bytes(response.content)

fpc_root = ET.fromstring(fpc_path.read_bytes())
fpc_nodes = {
    node.attrib["id"]: (float(node.attrib["lon"]), float(node.attrib["lat"]))
    for node in fpc_root.findall("node")
}
fpc_rows = []
for way in fpc_root.findall("way"):
    tags = {tag.attrib["k"]: tag.attrib["v"] for tag in way.findall("tag")}
    if "building" not in tags or tags.get("addr:housenumber") != "96":
        continue
    refs = [node.attrib["ref"] for node in way.findall("nd")]
    coords = [fpc_nodes[ref] for ref in refs if ref in fpc_nodes]
    if len(coords) >= 4 and coords[0] == coords[-1]:
        fpc_rows.append({"osm_id": int(way.attrib["id"]), "geometry": Polygon(coords)})

fpc_buildings = gpd.GeoDataFrame(fpc_rows, geometry="geometry", crs=4326).to_crs(31370)
assert len(fpc_buildings) == 2

# Subdivide the footprint based on a ground-truth image.
FPC_HEIGHT_BANDS = [
    (287.0, 289.4, 28.0),
    (289.4, 290.8, 12.0),
    (290.8, 292.4, 16.0),
    (292.4, 295.0, 12.0),
    (295.0, 299.0, 20.0),
]
photo_site = observers.loc[observers["name"] == "Nieuw Zuid"].iloc[0]
sector_radius_m = 2000.0
fpc_models = []
for azimuth_min, azimuth_max, height_m in FPC_HEIGHT_BANDS:
    angles = np.radians([azimuth_min, azimuth_max])
    sector = Polygon([
        (photo_site["x_l72"], photo_site["y_l72"]),
        *zip(
            photo_site["x_l72"] + sector_radius_m * np.sin(angles),
            photo_site["y_l72"] + sector_radius_m * np.cos(angles),
        ),
    ])
    for footprint in fpc_buildings.geometry:
        segment = footprint.intersection(sector)
        if not segment.is_empty:
            fpc_models.append((segment, height_m))

def model_footprint_horizon(models, observer):
    azimuth_grid = np.arange(
        solar.WEDGE_AZ_MIN,
        solar.WEDGE_AZ_MAX + horizon.DEFAULT_AZ_STEP / 2,
        horizon.DEFAULT_AZ_STEP,
    )
    roofline = np.full(azimuth_grid.shape, -90.0)
    roof_distance = np.full(azimuth_grid.shape, np.nan)
    roof_height = np.full(azimuth_grid.shape, np.nan)
    with rasterio.open(DTM_VRT) as src:
        for footprint, building_height_m in models:
            window = from_bounds(*footprint.bounds, src.transform)
            window = window.round_offsets().round_lengths()
            ground = src.read(1, window=window, masked=True)
            transform = src.window_transform(window)
            inside = geometry_mask(
                [footprint.__geo_interface__], ground.shape, transform, invert=True
            )
            valid = inside & ~np.ma.getmaskarray(ground)
            if not valid.any():
                continue
            rows, cols = np.where(valid)
            xs, ys = rasterio.transform.xy(transform, rows, cols, offset="center")
            azimuths, distances = aoi.bearing_distance(
                observer["x_l72"], observer["y_l72"], np.asarray(xs), np.asarray(ys)
            )
            roof_taw = float(np.ma.median(ground[valid])) + building_height_m
            angles = np.degrees(np.arctan2(
                roof_taw - observer["z_eye_taw"]
                - horizon.curvature_drop(distances),
                distances,
            ))
            bins = np.rint(
                (azimuths - solar.WEDGE_AZ_MIN) / horizon.DEFAULT_AZ_STEP
            ).astype(int)
            keep = (bins >= 0) & (bins < azimuth_grid.size)
            for bin_index in np.unique(bins[keep]):
                choices = np.flatnonzero(keep & (bins == bin_index))
                choice = choices[np.argmax(angles[choices])]
                if angles[choice] > roofline[bin_index]:
                    roofline[bin_index] = angles[choice]
                    roof_distance[bin_index] = distances[choice]
                    roof_height[bin_index] = roof_taw
    return azimuth_grid, roofline, roof_distance, roof_height

fpc_updates = {}
fpc_checks = []
eclipse_track = track[
    (track["utc_hours"] + solar.UTC_OFFSET_HOURS >= C1_CEST)
    & (track["utc_hours"] + solar.UTC_OFFSET_HOURS <= C4_CEST)
]
for _, observer in observers.iterrows():
    update = model_footprint_horizon(fpc_models, observer)
    fpc_updates[observer["name"]] = update
    azimuth_grid, roofline, _, _ = update
    present = (
        (roofline > -90)
        & (azimuth_grid >= eclipse_track["azimuth_deg"].min())
        & (azimuth_grid <= eclipse_track["azimuth_deg"].max())
    )
    if present.any():
        sun_height = np.interp(
            azimuth_grid[present],
            eclipse_track["azimuth_deg"],
            eclipse_track["altitude_deg"],
        )
        minimum_clearance = float(np.min(sun_height - roofline[present]))
        fpc_checks.append({
            "site tested": observer["name"],
            "FPC height range used (m)": "12-28",
            "minimum Sun clearance above roofline (degrees)": round(minimum_clearance, 2),
            "intersects late eclipse path": minimum_clearance <= 0,
        })

assert fpc_checks
pd.DataFrame(fpc_checks)


,site tested,FPC height range used (m),minimum Sun clearance above roofline (degrees),intersects late eclipse path
0,Nieuw Zuid,12-28,-0.35,True


## Horizon profiles and visibility


In [7]:
# Cast a horizon profile from each observer and calculate eclipse visibility.
profiles = {}
results = []

for _, observer in observers.iterrows():
    profile = horizon.compute_horizon(
        DSM_VRT,
        observer["x_l72"],
        observer["y_l72"],
        observer["z_eye_taw"],
        classes_path=CLASSES,
        exclude_classes=(surface.CLASS_WATER,),
        ray_m=RAY_M,
        name=observer["name"],
    )
    # Replace older LiDAR heights with the current buildings where they rise higher.
    if observer["name"] == "Scheldekaaien Zuid":
        raised = development_horizon > profile.horizon_deg
        profile.horizon_deg[raised] = development_horizon[raised]
        profile.distance_m[raised] = development_distance[raised]
        profile.height_taw[raised] = development_height[raised]
        profile.class_code[raised] = surface.CLASS_BUILDING
    _, fpc_horizon, fpc_distance, fpc_height = fpc_updates[observer["name"]]
    raised = fpc_horizon > profile.horizon_deg
    profile.horizon_deg[raised] = fpc_horizon[raised]
    profile.distance_m[raised] = fpc_distance[raised]
    profile.height_taw[raised] = fpc_height[raised]
    profile.class_code[raised] = surface.CLASS_BUILDING
    profiles[observer["name"]] = profile

    visibility = horizon.visibility(
        profile,
        track,
        max_eclipse_cest=MAX_CEST,
        eclipse_window_cest=ECLIPSE_WINDOW,
    )
    visibility["z_mode"] = observer["z_mode"]
    visibility["median_horizon_deg"] = round(
        float(np.median(profile.horizon_deg)), 2
    )
    visibility["max_horizon_deg"] = round(
        float(profile.horizon_deg.max()), 2
    )
    results.append(visibility)

results = pd.DataFrame(results)
results.sort_values("clearance_at_max", ascending=False)


,name,z_eye_taw,sun_alt_at_max,horizon_at_max,clearance_at_max,visible_at_max,first_visible_centre,last_visible_centre,minutes_visible_centre,first_visible_upper_limb,last_visible_upper_limb,minutes_visible_upper_limb,eclipse_first_cest,eclipse_last_cest,eclipse_minutes,eclipse_clipped_by_wedge,z_mode,median_horizon_deg,max_horizon_deg
4,MAS panoramic platform,70.28,7.7,-0.30,8.00,True,19:14,21:11,117.2,19:14,21:11,117.3,19:18,21:05,106.6,False,surface,-0.27,0.55
3,Droogdokkenpark,5.90,7.7,0.29,7.41,True,19:14,20:59,105.1,19:14,21:02,107.7,19:18,21:02,103.2,False,ground,0.36,1.85
1,Scheldekaaien Zuid,8.40,7.7,0.41,7.29,True,19:14,21:00,106.7,19:14,21:03,109.5,19:18,21:03,105.1,False,ground,0.55,1.82
0,Nieuw Zuid,8.45,7.7,1.05,6.65,True,19:14,21:00,105.9,19:14,21:02,107.8,19:18,21:02,103.3,False,ground,0.96,1.84
2,"Wandelterras, Het Steen",13.59,7.7,1.76,5.94,True,19:14,20:51,96.8,19:14,20:51,97.3,19:18,20:51,92.9,False,ground,1.69,2.86


## Obstruction classes


In [8]:
# Summarize the vegetation, building, and unknown features controlling each skyline.
rows = []
for name, p in profiles.items():
    for code in (surface.CLASS_VEGETATION, surface.CLASS_BUILDING,
                 surface.CLASS_UNKNOWN):
        sel = p.class_code == code
        if not sel.any():
            continue
        h = p.horizon_deg[sel]
        rows.append({
            "observer": name,
            "class": surface.CLASS_LABELS[code],
            "bearings": int(sel.sum()),
            "median_deg": round(float(np.median(h)), 2),
            "max_deg": round(float(h.max()), 2),
            "above maximum": int((h > tmax["altitude_deg"]).sum()),
            "above 3 deg": int((h > 3.0).sum()),
        })

by_class = pd.DataFrame(rows)
by_class


,observer,class,bearings,median_deg,max_deg,above maximum,above 3 deg
0,Nieuw Zuid,vegetation,133,0.74,1.47,0,0
1,Nieuw Zuid,building,97,1.09,1.84,0,0
2,Nieuw Zuid,"tall, unattributed",1,0.66,0.66,0,0
3,Scheldekaaien Zuid,vegetation,179,0.56,1.82,0,0
4,Scheldekaaien Zuid,building,52,0.52,0.81,0,0
5,"Wandelterras, Het Steen",vegetation,117,1.56,2.86,0,0
6,"Wandelterras, Het Steen",building,114,1.85,2.84,0,0
7,Droogdokkenpark,vegetation,128,0.75,1.85,0,0
8,Droogdokkenpark,building,27,0.25,1.14,0,0
9,Droogdokkenpark,"tall, unattributed",70,0.23,0.45,0,0


## Outputs


In [9]:
# Save the horizon profiles, visibility results, class summary, and analysis settings.
out_dir = PROCESSED / "horizon_profiles"
out_dir.mkdir(parents=True, exist_ok=True)
for old in out_dir.glob("*.csv"):
    old.unlink()

for name, p in profiles.items():
    slug = "_".join(name.lower().replace(",", "").split())
    p.to_frame().to_csv(out_dir / f"{slug}.csv", index=False)

results.to_csv(PROCESSED / "04_visibility.csv", index=False)
by_class.to_csv(PROCESSED / "04_horizon_by_class.csv", index=False)

summary = {
    "ray_m": RAY_M,
    "az_range": [solar.WEDGE_AZ_MIN, solar.WEDGE_AZ_MAX],
    "curvature_k": horizon.K_REFRACTION,
    "excluded_classes": ["water"],
    "max_eclipse": {"cest": contact_data["maximum_hhmm"],
                    "altitude_deg": round(float(tmax["altitude_deg"]), 2),
                    "azimuth_deg": round(float(tmax["azimuth_deg"]), 2)},
    "observers": len(profiles),
    "best_by_clearance": results.sort_values(
        "clearance_at_max", ascending=False).iloc[0]["name"],
}
(PROCESSED / "04_horizon_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(f"\nprofiles -> {out_dir.relative_to(PROJECT_ROOT)}")


{
  "ray_m": 10000,
  "az_range": [
    273.0,
    296.0
  ],
  "curvature_k": 0.13,
  "excluded_classes": [
    "water"
  ],
  "max_eclipse": {
    "cest": "20:13",
    "altitude_deg": 7.7,
    "azimuth_deg": 284.22
  },
  "observers": 5,
  "best_by_clearance": "MAS panoramic platform"
}

profiles -> data\processed\horizon_profiles
